In [ ]:
%pip install -r requirements.txt

In [ ]:
# 設定參數與本地影片資料夾
from pathlib import Path
from datetime import datetime
import re

MODEL_SIZE = "large-v3"   # tiny / base / small / medium / large-v3
LANGUAGE = "zh"           # None = 自動偵測 zh:中文 en:英文 ja:日語
TASK = "transcribe"       # transcribe:直接輸出 / translate:翻譯成英文後輸出

WORKSPACE_DIR = Path.cwd().resolve()
VIDEO_INPUT_DIR = (WORKSPACE_DIR / "local_videos").resolve()
BASE_OUTPUT_DIR = (WORKSPACE_DIR / "projects").resolve()
SHARED_MODEL_DIR = (WORKSPACE_DIR / "model_cache").resolve()

VIDEO_INPUT_DIR.mkdir(parents=True, exist_ok=True)
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SHARED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 指定要轉的檔名（留空 = 自動抓資料夾第一個影片）
LOCAL_VIDEO_NAME = ""

SUPPORTED_EXTS = (".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v")
video_files = sorted([p for p in VIDEO_INPUT_DIR.iterdir() if p.suffix.lower() in SUPPORTED_EXTS])

if not video_files:
    raise FileNotFoundError(
        f"找不到影片。請把影片放到: {VIDEO_INPUT_DIR}\n"
        f"支援格式: {', '.join(SUPPORTED_EXTS)}"
    )

if LOCAL_VIDEO_NAME:
    VIDEO_PATH = (VIDEO_INPUT_DIR / LOCAL_VIDEO_NAME).resolve()
    if not VIDEO_PATH.exists():
        raise FileNotFoundError(f"指定影片不存在: {VIDEO_PATH}")
else:
    VIDEO_PATH = video_files[0].resolve()
    print(f"ℹ️ 未指定 LOCAL_VIDEO_NAME，自動選擇: {VIDEO_PATH.name}")

def sanitize_windows_name(name: str, max_len: int = 60) -> str:
    cleaned = re.sub(r"[<>:\"/\\|?*]", "_", name).strip().rstrip(".")
    cleaned = re.sub(r"\s+", " ", cleaned)
    return (cleaned[:max_len].strip() or "untitled")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
safe_title = sanitize_windows_name(VIDEO_PATH.stem)
PROJECT_DIR = (BASE_OUTPUT_DIR / f"{timestamp}_{safe_title}").resolve()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_MEDIA_PATH = VIDEO_PATH
OUTPUT_TXT_PATH = (PROJECT_DIR / "output.txt").resolve()
OUTPUT_SRT_PATH = (PROJECT_DIR / "output.srt").resolve()
TRANSLATED_EN_PATH = (PROJECT_DIR / "translated_en.txt").resolve()
SOURCE_INFO_PATH = (PROJECT_DIR / "source.txt").resolve()

SOURCE_INFO_PATH.write_text(
    f"SOURCE_TYPE=local_file\n"
    f"PATH={VIDEO_PATH}\n",
    encoding="utf-8"
 )

print(f"影片來源資料夾: {VIDEO_INPUT_DIR}")
print(f"目前影片: {VIDEO_PATH}")
print(f"專案資料夾: {PROJECT_DIR}")
print(f"共用模型資料夾: {SHARED_MODEL_DIR}")

In [ ]:
# 準備本地媒體（絕對路徑）
if not INPUT_MEDIA_PATH.exists():
    raise FileNotFoundError(f"❌ 找不到本地影片: {INPUT_MEDIA_PATH}")

size_mb = INPUT_MEDIA_PATH.stat().st_size / (1024 * 1024)
print(f"✅ 本地影片準備完成: {INPUT_MEDIA_PATH}")
print(f"📦 檔案大小: {size_mb:.2f} MB")

In [ ]:
# 轉錄（本地影片 + 共用模型快取）
import os
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

def resolve_repo_id(model_size: str) -> str:
    if "/" in model_size:
        return model_size
    return f"Systran/faster-whisper-{model_size}"

repo_id = resolve_repo_id(MODEL_SIZE)
local_model_dir = (SHARED_MODEL_DIR / repo_id.replace("/", "__")).resolve()
local_model_dir.mkdir(parents=True, exist_ok=True)

model_path = snapshot_download(
    repo_id=repo_id,
    local_dir=str(local_model_dir),
 )

model = WhisperModel(model_path, compute_type="float16")

segments_iter, info = model.transcribe(
    str(INPUT_MEDIA_PATH),
    beam_size=5,
    vad_filter=True,
    language=LANGUAGE,
    task=TASK,
 )

segments = []
for seg in tqdm(segments_iter, desc="轉錄中", unit="segment"):
    segments.append(seg)

print(f"✅ 轉錄完成，共 {len(segments)} 段: {OUTPUT_TXT_PATH}")

In [ ]:
# 儲存逐字稿（絕對路徑）
with open(OUTPUT_TXT_PATH, "w", encoding="utf-8") as f:
    for seg in segments:
        f.write(seg.text.strip() + "\n")

print(f"✅ 逐字稿已生成: {OUTPUT_TXT_PATH}")

In [ ]:
# 可選：產生清理版逐字稿（排除台語/不需要段落）
import re

# 方式1：用時間區間排除（秒）
# 例：[(30, 95), (320, 410)] 代表排除 00:30-01:35、05:20-06:50
EXCLUDE_TIME_RANGES = [(240, 307)]  # 04:00~05:07

# 方式2：用關鍵字排除（命中就整段刪除）
# 先留空；若你知道常出現的詞可再加入
EXCLUDE_KEYWORDS = []

def overlap(a_start, a_end, b_start, b_end):
    return not (a_end <= b_start or a_start >= b_end)

clean_segments = []
removed_count = 0

for seg in segments:
    text = seg.text.strip()
    remove_by_time = any(
        overlap(seg.start, seg.end, s, e) for s, e in EXCLUDE_TIME_RANGES
    )
    remove_by_keyword = any(
        re.search(re.escape(k), text, flags=re.IGNORECASE)
        for k in EXCLUDE_KEYWORDS if k.strip()
    )

    if remove_by_time or remove_by_keyword:
        removed_count += 1
        continue

    clean_segments.append(seg)

CLEAN_OUTPUT_TXT_PATH = (PROJECT_DIR / "output_clean.txt").resolve()
with open(CLEAN_OUTPUT_TXT_PATH, "w", encoding="utf-8") as f:
    for seg in clean_segments:
        f.write(seg.text.strip() + "\n")

print(f"✅ 清理版逐字稿已生成: {CLEAN_OUTPUT_TXT_PATH}")
print(f"移除段落數: {removed_count} / 原始段落數: {len(segments)}")

In [ ]:
# 可選：生成 SRT 字幕檔（絕對路徑）
def format_time(t):
    h = int(t//3600)
    m = int((t%3600)//60)
    s = int(t%60)
    ms = int((t-int(t))*1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

with open(OUTPUT_SRT_PATH, "w", encoding="utf-8") as f:
    for i, seg in enumerate(segments, 1):
        f.write(f"{i}\n")
        f.write(f"{format_time(seg.start)} --> {format_time(seg.end)}\n")
        f.write(seg.text.strip() + "\n\n")

print(f"✅ 字幕檔已生成: {OUTPUT_SRT_PATH}")

In [ ]:
# 可選: 生成翻譯後的英文文本（絕對路徑）
segments_en, _ = model.transcribe(
    str(INPUT_MEDIA_PATH),
    task="translate",
 )

segments_en = list(segments_en)

with open(TRANSLATED_EN_PATH, "w", encoding="utf-8") as f:
    for seg in segments_en:
        f.write(seg.text.strip() + "\n")

print(f"✅ 英文翻譯完成: {TRANSLATED_EN_PATH}")